# Experiment 3: Pause after the think part (MLP), change its output, resume

Each of GPT-2's 12 blocks has two parts that run in order:
1. **Look-around part (attention):** every word gathers information from the words before it.
2. **Think part (MLP):** every word is processed **on its own**, using knowledge stored in the model.

Each part **adds** its result to the running table of numbers.

In this notebook we:
1. Check that running each block in two parts gives exactly the normal answer.
2. **PAUSE** inside a block, right after its think part (before its result is added), and **save** the result.
3. Show which words that result "votes" for.
4. **MODIFY** the saved think result.
5. **RESUME** by adding it to the table and running the remaining blocks, **without rerunning** anything earlier.
6. Compare normal and modified predictions.

We do this at two places:
- **Part A:** block 0, row of the word ' Paris'
- **Part B:** block 8, row of the last word ' speak'

## Step 0: Install (Colab only)
Tested with PyTorch 2.14.1 and Transformers 5.20.2. Uncomment only if something breaks.

In [1]:
# !pip install -q transformers==5.20.2

## Step 1: Load GPT-2 small

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.masking_utils import create_causal_mask

torch.set_grad_enabled(False)  # prediction only, no training
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device).eval()
gpt2 = model.transformer
NUM_BLOCKS = len(gpt2.h)

print("Device:", device)
print("Number of blocks:", NUM_BLOCKS)
print("Think part of one block:", gpt2.h[0].mlp)

/Users/arinpatel/Desktop/BTP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1023 00:24:27.837000 55422 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 8551.31it/s]

Device: cpu
Number of blocks: 12
Think part of one block: GPT2MLP(
  (c_fc): Conv1D(nf=3072, nx=768)
  (c_proj): Conv1D(nf=768, nx=3072)
  (act): NewGELUActivation()
  (dropout): Dropout(p=0.1, inplace=False)
)


## Step 2: Normal run (no changes)

In [3]:
PROMPT = "I live in Paris, so I speak"
OTHER_PROMPT = "I live in Berlin, so I speak"
input_ids = tokenizer(PROMPT, return_tensors="pt").input_ids.to(device)
other_ids = tokenizer(OTHER_PROMPT, return_tensors="pt").input_ids.to(device)
assert other_ids.shape == input_ids.shape
words = [tokenizer.decode(t) for t in input_ids[0]]
print("Word-pieces:", words)

PARIS_POS, LAST_POS = 3, len(words) - 1
FRENCH, GERMAN = tokenizer.encode(" French")[0], tokenizer.encode(" German")[0]

def top_predictions(logits, k=5):
    probs = logits[0, -1].softmax(-1)
    values, indices = probs.topk(k)
    return [(tokenizer.decode(i), v.item()) for i, v in zip(indices, values)]

def chances(logits):
    p = logits[0, -1].softmax(-1)
    return tokenizer.decode(p.argmax()), p[FRENCH].item(), p[GERMAN].item()

def compare(new_logits, title="Modified model"):
    """Print the normal and new top 5 side by side, plus the chance of ' French' and ' German'."""
    print(f"{'Normal model':<26}{title}")
    for (w1, p1), (w2, p2) in zip(top_predictions(normal_logits), top_predictions(new_logits)):
        print(f"  {w1!r:12} {p1:6.2%}      {w2!r:12} {p2:6.2%}")
    _, f0, g0 = chances(normal_logits)
    _, f1, g1 = chances(new_logits)
    print(f"\n  ' French': {f0:6.2%} -> {f1:6.2%}")
    print(f"  ' German': {g0:6.2%} -> {g1:6.2%}")

normal_logits = model(input_ids, use_cache=False).logits
print("\nNormal model - top 5 next words:")
for word, p in top_predictions(normal_logits):
    print(f"  {word!r:12} {p:6.2%}")

Word-pieces: ['I', ' live', ' in', ' Paris', ',', ' so', ' I', ' speak']



Normal model - top 5 next words:
  ' French'    63.05%
  ' English'    4.93%
  ' to'         3.16%
  ' a'          2.94%
  ' with'       1.76%


## Step 3: Run each block in two parts
- `look_part(i, state)` runs the look-around part of block `i` and adds its result to the table.
- `think_part(i, state)` runs the think part of block `i` and returns its result **without adding it**.
- `run_until_think(ids, layer)` runs everything up to and including the think part of block `layer`, then **PAUSES**.
  It returns a **checkpoint**: the table just before the think result is added, plus the think result.
- `resume_after_think(checkpoint)` **RESUMES**: adds the think result, then runs the remaining blocks.

In [4]:
def embed(input_ids):
    positions = torch.arange(input_ids.shape[1], device=input_ids.device).unsqueeze(0)
    return gpt2.wte(input_ids) + gpt2.wpe(positions)

def look_part(i, state):
    """Look-around (attention) part of block i, with its result added to the table."""
    block = gpt2.h[i]
    positions = torch.arange(state.shape[1], device=state.device).unsqueeze(0)
    # rule that each word can only look at earlier words (same as the normal model)
    mask = create_causal_mask(config=model.config, inputs_embeds=state, attention_mask=None,
                              past_key_values=None, position_ids=positions)
    look_result, _ = block.attn(block.ln_1(state), attention_mask=mask, position_ids=positions)
    return state + look_result

def think_part(i, state):
    """Think (MLP) part of block i. Returns its result; does NOT add it to the table."""
    block = gpt2.h[i]
    return block.mlp(block.ln_2(state))

def final_logits(state):
    return model.lm_head(gpt2.ln_f(state))

def run_model(input_ids, edits=None):
    """Full run, part by part. `edits` = {block number: function that changes its think result}."""
    edits = edits or {}
    state = embed(input_ids)
    for i in range(NUM_BLOCKS):
        state = look_part(i, state)
        think_result = think_part(i, state)
        if i in edits:
            think_result = edits[i](think_result)
        state = state + think_result
    return final_logits(state)

def run_until_think(input_ids, layer):
    """Run blocks 0..layer-1 fully, then block `layer` up to its think result, then PAUSE."""
    state = embed(input_ids)
    for i in range(layer):
        state = look_part(i, state)
        state = state + think_part(i, state)
    state = look_part(layer, state)
    return {"layer": layer, "before_think": state, "think_result": think_part(layer, state)}

def resume_after_think(checkpoint):
    """RESUME: add the (possibly changed) think result, then run all later blocks."""
    state = checkpoint["before_think"] + checkpoint["think_result"]
    for i in range(checkpoint["layer"] + 1, NUM_BLOCKS):
        state = look_part(i, state)
        state = state + think_part(i, state)
    return final_logits(state)

def copy_ckpt(checkpoint, **changes):
    """Separate copy of a checkpoint, optionally with some parts replaced."""
    new = {k: (v.clone() if torch.is_tensor(v) else v) for k, v in checkpoint.items()}
    new.update(changes)
    return new

def set_row(result, pos, new_row):
    """Copy of `result` with the row at `pos` replaced."""
    result = result.clone()
    result[0, pos] = new_row
    return result

## Step 4: A counter that records which parts actually run
`look 5` means the look-around part of block 5 ran; `think 5` means the think part of block 5 ran.

In [5]:
for h in globals().get("counter_hooks", []):  # safe to rerun this cell
    h.remove()

parts_run = []
counter_hooks = []
for i, block in enumerate(gpt2.h):
    counter_hooks.append(block.attn.register_forward_hook(lambda m, inp, out, i=i: parts_run.append(f"look {i}")))
    counter_hooks.append(block.mlp.register_forward_hook(lambda m, inp, out, i=i: parts_run.append(f"think {i}")))
print("Counter attached to", len(counter_hooks), "parts (12 blocks x 2 parts)")

def expected_after(layer):
    return [f"{p} {i}" for i in range(layer + 1, NUM_BLOCKS) for p in ("look", "think")]

Counter attached to 24 parts (12 blocks x 2 parts)


## Step 5: Check that the two-part run matches the normal model

In [6]:
parts_run.clear()
split_logits = run_model(input_ids)

diff = (split_logits - normal_logits).abs().max().item()
print("Parts run:", len(parts_run), "->", parts_run[:4], "...", parts_run[-2:])
print("Largest difference from normal model:", diff)
assert diff < 1e-4, "Two-part run does not match the normal model"
print("OK: two-part run matches the normal model")

Parts run: 24 -> ['look 0', 'think 0', 'look 1', 'think 1'] ... ['look 11', 'think 11']
Largest difference from normal model: 0.0
OK: two-part run matches the normal model


## Step 6: A rough way to read a think result
A think result is 768 numbers per word, which we cannot read directly.
As a rough guide, we pass one row straight through the model's final "numbers to words" step and list the words it pushes up the most.
This is only an approximation (it skips the final cleanup step), but it hints at what the result is about.

In [7]:
def word_votes(row, k=8):
    scores = model.lm_head(row)
    return [tokenizer.decode(t) for t in scores.topk(k).indices]

---
# Part A: block 0, row of ' Paris'

## Step 7: PAUSE after the think part of block 0

In [8]:
PAUSE_A = 0

parts_run.clear()
ckpt_a = run_until_think(input_ids, PAUSE_A)  # <-- PAUSE happens here
saved_a = copy_ckpt(ckpt_a)                   # <-- SAVE a separate copy

print(f"PAUSED inside block {PAUSE_A}, after its think part (result not added yet)")
print("Parts run so far:", parts_run)
print("Think result size:", tuple(saved_a["think_result"].shape), " (1 sentence, 8 word-pieces, 768 numbers each)")

other_a = run_until_think(other_ids, PAUSE_A)
print("\nWords pushed up by the think result of ' Paris' :", word_votes(saved_a["think_result"][0, PARIS_POS]))
print("Words pushed up by the think result of ' Berlin':", word_votes(other_a["think_result"][0, PARIS_POS]))

PAUSED inside block 0, after its think part (result not added yet)
Parts run so far: ['look 0', 'think 0']
Think result size: (1, 8, 768)  (1 sentence, 8 word-pieces, 768 numbers each)

Words pushed up by the think result of ' Paris' : [' Hilton', ',', '-', 'ian', 'ienne', ' Paris', ' and', 'ien']
Words pushed up by the think result of ' Berlin': [',', 'er', '-', '.', ' and', ' D', ' to', 'ers']


## Step 8: RESUME with the saved result unchanged (check)

In [9]:
parts_run.clear()
resumed = resume_after_think(copy_ckpt(saved_a))  # <-- RESUME

print("RESUMED. Parts run:", parts_run[:4], "...", parts_run[-2:], f"({len(parts_run)} parts)")
print("Largest difference from normal model:", (resumed - normal_logits).abs().max().item())
assert parts_run == expected_after(PAUSE_A), "Something earlier was rerun!"
assert torch.allclose(resumed, normal_logits, atol=1e-4)
print("OK: resuming gives the normal answer, and block 0 was not rerun")

RESUMED. Parts run: ['look 1', 'think 1', 'look 2', 'think 2'] ... ['look 11', 'think 11'] (22 parts)
Largest difference from normal model: 0.0
OK: resuming gives the normal answer, and block 0 was not rerun


## Step 9: MODIFY — replace the ' Paris' think result with the ' Berlin' one
Only the row of ' Paris' in the think result changes. The original word ' Paris' is still in the table.

In [10]:
new_result = set_row(saved_a["think_result"], PARIS_POS, other_a["think_result"][0, PARIS_POS])  # <-- MODIFY

parts_run.clear()
logits_a_swap = resume_after_think(copy_ckpt(saved_a, think_result=new_result))  # <-- RESUME
assert parts_run == expected_after(PAUSE_A)
compare(logits_a_swap, "Berlin think result (block 0)")

Normal model              Berlin think result (block 0)
  ' French'    63.05%      ' German'    54.21%
  ' English'    4.93%      ' English'    5.40%
  ' to'         3.16%      ' French'     5.37%
  ' a'          2.94%      ' a'          4.27%
  ' with'       1.76%      ' to'         3.92%

  ' French': 63.05% ->  5.37%
  ' German':  0.62% -> 54.21%


## Step 10: MODIFY — switch off the ' Paris' think result (set to zero)

In [11]:
new_result = set_row(saved_a["think_result"], PARIS_POS, 0)  # <-- MODIFY

parts_run.clear()
logits_a_off = resume_after_think(copy_ckpt(saved_a, think_result=new_result))  # <-- RESUME
assert parts_run == expected_after(PAUSE_A)
compare(logits_a_off, "Switched off (block 0)")

Normal model              Switched off (block 0)
  ' French'    63.05%      ' French'    72.10%
  ' English'    4.93%      ' English'    2.32%
  ' to'         3.16%      ' to'         2.22%
  ' a'          2.94%      ' a'          1.82%
  ' with'       1.76%      ' with'       1.73%

  ' French': 63.05% -> 72.10%
  ' German':  0.62% ->  0.28%


---
# Part B: block 8, row of ' speak'

## Step 11: PAUSE after the think part of block 8

In [12]:
PAUSE_B = 8

parts_run.clear()
ckpt_b = run_until_think(input_ids, PAUSE_B)  # <-- PAUSE happens here
saved_b = copy_ckpt(ckpt_b)                   # <-- SAVE

print(f"PAUSED inside block {PAUSE_B}, after its think part")
print("Parts run so far:", parts_run[-3:], f"... ({len(parts_run)} parts in total)")

other_b = run_until_think(other_ids, PAUSE_B)
print("\nWords pushed up by the think result of ' speak' (Paris sentence) :", word_votes(saved_b["think_result"][0, LAST_POS]))
print("Words pushed up by the think result of ' speak' (Berlin sentence):", word_votes(other_b["think_result"][0, LAST_POS]))

PAUSED inside block 8, after its think part
Parts run so far: ['think 7', 'look 8', 'think 8'] ... (18 parts in total)

Words pushed up by the think result of ' speak' (Paris sentence) : [' French', ' English', ' Et', ' antiqu', ' a', ' the', ' to', ' France']


Words pushed up by the think result of ' speak' (Berlin sentence): [' English', ' Korean', ' Spanish', ' German', ' to', ' Italian', 'common', ' english']


## Step 12: MODIFY — make the ' speak' think result weaker, stronger, or reversed
We multiply that one row by a number: 0 = switched off, 1 = unchanged, 2 = twice as strong, -1 = reversed.
Each line is a separate resume from the same saved checkpoint.

In [13]:
print(f"{'Multiply by':<13}{'top word':<12}{'French':>8}{'German':>8}   parts run after resume")
for k in [0, 0.5, 1, 2, 3, -1]:
    row = saved_b["think_result"][0, LAST_POS] * k
    parts_run.clear()
    logits = resume_after_think(copy_ckpt(saved_b, think_result=set_row(saved_b["think_result"], LAST_POS, row)))
    assert parts_run == expected_after(PAUSE_B)
    top, f, g = chances(logits)
    print(f"{k:<13}{top!r:<12}{f:>8.1%}{g:>8.1%}   {parts_run[0]} ... {parts_run[-1]}")

Multiply by  top word      French  German   parts run after resume


0            ' French'      34.1%    0.6%   look 9 ... think 11
0.5          ' French'      54.0%    0.7%   look 9 ... think 11
1            ' French'      63.0%    0.6%   look 9 ... think 11


2            ' French'      68.0%    0.6%   look 9 ... think 11
3            ' French'      68.6%    0.6%   look 9 ... think 11
-1           ' from'         1.5%    0.1%   look 9 ... think 11


## Step 13: MODIFY — replace the ' speak' think result with the Berlin sentence's one

In [14]:
new_result = set_row(saved_b["think_result"], LAST_POS, other_b["think_result"][0, LAST_POS])  # <-- MODIFY

parts_run.clear()
logits_b_swap = resume_after_think(copy_ckpt(saved_b, think_result=new_result))  # <-- RESUME
assert parts_run == expected_after(PAUSE_B)
compare(logits_b_swap, "Berlin think result (block 8)")

Normal model              Berlin think result (block 8)
  ' French'    63.05%      ' French'    57.34%
  ' English'    4.93%      ' English'    5.59%
  ' to'         3.16%      ' to'         3.89%
  ' a'          2.94%      ' a'          3.44%
  ' with'       1.76%      ' Spanish'    2.35%

  ' French': 63.05% -> 57.34%
  ' German':  0.62% ->  0.76%


---
## Step 14: Try the changes in every block, one block at a time

In [15]:
other_think = {i: run_until_think(other_ids, i)["think_result"] for i in range(NUM_BLOCKS)}

print(f"{'':<6}{'--- row of Paris ---':^32}{'--- row of speak ---':^32}")
print(f"{'Block':<6}{'Berlin version':>16}{'Switched off':>16}{'Berlin version':>16}{'Switched off':>16}")
print(f"{'':<6}" + f"{'French/German':>16}" * 4)
for layer in range(NUM_BLOCKS):
    ckpt = run_until_think(input_ids, layer)
    line = f"{layer:<6}"
    for pos in (PARIS_POS, LAST_POS):
        for new_row in (other_think[layer][0, pos], 0):
            _, f, g = chances(resume_after_think(copy_ckpt(ckpt, think_result=set_row(ckpt["think_result"], pos, new_row))))
            line += f"{f:>9.0%}/{g:<5.0%} "
    print(line)

            --- row of Paris ---            --- row of speak ---      
Block   Berlin version    Switched off  Berlin version    Switched off
         French/German   French/German   French/German   French/German


0            5%/54%         72%/0%          63%/1%          34%/1%    


1           64%/1%          66%/1%          62%/1%          60%/1%    


2           64%/1%          64%/1%          64%/1%          58%/1%    
3           67%/1%          66%/1%          64%/1%          57%/1%    


4           63%/1%          67%/1%          64%/1%          56%/1%    


5           62%/1%          64%/1%          62%/1%          58%/1%    
6           62%/1%          65%/1%          64%/1%          54%/1%    


7           64%/1%          65%/1%          63%/1%          57%/1%    


8           61%/1%          61%/1%          57%/1%          34%/1%    
9           58%/1%          58%/1%          62%/1%          35%/1%    
10          65%/1%          63%/1%          68%/1%          54%/0%    


11          63%/1%          63%/1%          41%/0%          89%/0%    


## Step 15: Change several blocks together
We pause after the think part of **each** listed block in the same run and switch off the chosen row every time.

In [16]:
def switch_off(pos):
    return lambda result: set_row(result, pos, 0)

for name, pos, layers in [("' Paris'", PARIS_POS, range(0, 8)), ("' speak'", LAST_POS, range(8, 12))]:
    parts_run.clear()
    logits = run_model(input_ids, edits={i: switch_off(pos) for i in layers})
    assert len(parts_run) == 2 * NUM_BLOCKS, "Each part should run exactly once"
    print(f"Switch off think result of {name} in blocks {layers.start}-{layers.stop - 1}:")
    compare(logits, f"Switched off ({layers.start}-{layers.stop - 1})")
    print()

Switch off think result of ' Paris' in blocks 0-7:
Normal model              Switched off (0-7)
  ' French'    63.05%      ' English'   14.51%
  ' English'    4.93%      ' French'     7.60%
  ' to'         3.16%      ' to'         7.25%
  ' a'          2.94%      ' Spanish'    5.44%
  ' with'       1.76%      ' the'        4.13%

  ' French': 63.05% ->  7.60%
  ' German':  0.62% ->  1.07%

Switch off think result of ' speak' in blocks 8-11:
Normal model              Switched off (8-11)
  ' French'    63.05%      ' French'     8.41%
  ' English'    4.93%      ' English'    7.48%
  ' to'         3.16%      ' about'      3.33%
  ' a'          2.94%      ' my'         2.58%
  ' with'       1.76%      ' and'        2.18%

  ' French': 63.05% ->  8.41%
  ' German':  0.62% ->  0.36%

